In [62]:
import os
import csv
from datetime import datetime
from zoneinfo import ZoneInfo

csv_file_path = "orders_raw.csv"
new_file_path = "final_orders.csv"


with open(csv_file_path,"r",newline="") as file:
    read_data = csv.DictReader(file)
    # header_row = next(file)
    my_list = [row for row in read_data]
    # for row in read_data:
    #     yield row
    # print(my_list)
# remove invalid rows
# if timestamp is invalid
# if quantity cannot be converted to integer
# if price cannot be converted to number
# Log or print an error message
    def remove_invalid(data):
        list_remove_invalid = []
        for i in data:
            # print(i)
            try:
                i["timestamp"] = datetime.strptime(i["timestamp"], "%Y-%m-%dT%H:%M:%S").strftime("%Y-%m-%d %H:%M:%S")
                i['quantity'] = int(i['quantity'])
                i["price"] = float(i["price"])
                list_remove_invalid.append(i)
            except ValueError as e:
                print(f"Invalid data: {i}, Error: {e}")
                # j.append(i)
        return list_remove_invalid

            # my_list.pop(invalid_items)

    clean_data = remove_invalid(my_list)
    # print(clean_data)

    def cancelled(data):
        list_remove_cancelled = []
        for i in data:
            if i["status"] != 'cancelled':
                list_remove_cancelled.append(i)
        return list_remove_cancelled

    ignore_cancelled = cancelled(clean_data)
    # print(ignore_cancelled)
    # print(len(ignore_cancelled))
    
    def remove_duplicates(data):
        unique_id = set()
        list_remove_dup = []
        for i in data:
            id = i["order_id"]
            if id not in unique_id:
                unique_id.add(id)
                list_remove_dup.append(i)
        return list_remove_dup
    
    final_data = remove_duplicates(ignore_cancelled)
    # print(final_data)
    # print(len(final_data))

    def final_amount(data):
        for i in data:
            i["total"] = i["price"] * i["quantity"]
            # print(i)
        return data
    total_amount = final_amount(final_data)
    # print(total_amount)

    def convert_to_ISO(data):
        for i in data:
            dt = datetime.strptime(i["timestamp"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=ZoneInfo("Asia/Kolkata"))
            i["timestamp"] = dt.isoformat()
        return data
    after_conversion = convert_to_ISO(total_amount)
    print(after_conversion)

with open(new_file_path,"w",newline="") as file:
    write_data = csv.DictWriter(file,fieldnames=after_conversion[0].keys())
    write_data.writeheader()
    write_data.writerows(after_conversion)

Invalid data: {'order_id': '1005', 'customer': 'David', 'timestamp': 'invalid_timestamp', 'product': 'Laptop', 'quantity': '1', 'price': '70000', 'status': 'completed'}, Error: time data 'invalid_timestamp' does not match format '%Y-%m-%dT%H:%M:%S'
Invalid data: {'order_id': '1006', 'customer': 'Eva', 'timestamp': '2026-01-01 11:30:00', 'product': 'Mouse', 'quantity': 'invalid_qty', 'price': '500', 'status': 'completed'}, Error: invalid literal for int() with base 10: 'invalid_qty'
[{'order_id': '1001', 'customer': 'Alice', 'timestamp': '2026-01-01T10:00:00+05:30', 'product': 'Laptop', 'quantity': 1, 'price': 70000.0, 'status': 'completed', 'total': 70000.0}, {'order_id': '1002', 'customer': 'Bob', 'timestamp': '2026-01-01T10:05:00+05:30', 'product': 'Mouse', 'quantity': 2, 'price': 500.0, 'status': 'completed', 'total': 1000.0}, {'order_id': '1003', 'customer': 'Alice', 'timestamp': '2026-01-01T10:07:00+05:30', 'product': 'Keyboard', 'quantity': 1, 'price': 2000.0, 'status': 'complete